# EP06｜120Hz 和 60Hz 差在哪？為什麼高更新率滑起來比較順

像素腦底加！這是影片裡的「更新率模擬器」，按上方的 **執行階段 → 全部執行**（或每格按 ▶️）就能自己跑一遍。
我們模擬眼睛追著移動的東西看時，60Hz、120Hz、240Hz 和黑插幀各會拖出多長的影子；再算手指要等多久、電影和遊戲在不同更新率上的節奏。

⚠️ 這是簡化的示意模型，用來看懂原理，不是某一款真實螢幕的實測數值。

📺 影片：[youtube.com/@pixelbraintw](https://www.youtube.com/@pixelbraintw)

## 0. 準備：載入套件、下載一張貓咪照片（CC0 授權，作者 AdinaVoicu，Wikimedia Commons）

In [ ]:
import os, urllib.request
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

if not os.path.exists("cat.png"):
    urllib.request.urlretrieve("https://raw.githubusercontent.com/pixelbraintw/pixelbrain-code/main/ep03-oled/cat.png", "cat.png")
cat = np.array(Image.open("cat.png").convert("RGB").resize((360, 360))) / 255.0
speed = 1800                     # 每秒移動 1800 像素（大概是手指快速滑動）

## 1. 眼睛追著看，畫面卻停住：拖影有多長？
把 `speed` 改成你想試的速度，或把 `cases` 改成你螢幕的更新率。

In [ ]:
def smear(img, hz, lit=1.0):
    # 每張畫面亮著的這段時間，眼睛移動了幾個像素
    width = speed / hz * lit
    n = max(1, int(round(width)))
    out = np.zeros_like(img)
    for k in range(n):   # 眼睛往右移，畫面不動
        out += np.roll(img, -k + n // 2, axis=1)
    return out / n, width

cases = {"60Hz": (60, 1.0), "120Hz": (120, 1.0), "240Hz": (240, 1.0), "60Hz 黑插幀": (60, 0.25)}
fig, axes = plt.subplots(1, len(cases), figsize=(16, 4.5))
for ax, (name, (hz, lit)) in zip(axes, cases.items()):
    img, width = smear(cat, hz, lit)
    print(name, "拖影", round(width, 1), "像素")
    ax.imshow(img); ax.set_title(f"{hz}Hz lit={lit}: {width:.1f}px"); ax.axis("off")
plt.show()

## 2. 手指點下去，要等多久才換到下一張？

In [ ]:
rng = np.random.default_rng(0)
touch = rng.uniform(0, 1, 100_000)          # 10 萬次隨機時間點下去（秒）
for hz in (60, 120):
    frame = 1 / hz
    wait = (np.ceil(touch / frame) * frame - touch) * 1000   # 等到下一次換圖，換算成毫秒
    print(f"{hz}Hz 平均等 {wait.mean():.2f} 毫秒，最久 {wait.max():.2f} 毫秒")

## 3. 每秒 24 格的電影，放在 60Hz 和 120Hz 上
試試把 `(60, 120)` 改成 `(90, 144)`，看哪一個會整整齊齊。

In [ ]:
for hz in (60, 120):
    ticks = np.arange(hz)                         # 一秒內的每一次換圖
    film = np.floor(ticks * 24 / hz).astype(int)  # 這時電影放到第幾格
    counts = np.bincount(film)[:8]               # 每格被放了幾次
    print(f"{hz}Hz 每格放幾次：", counts.tolist())

## 4. 遊戲每秒畫 90 張，螢幕每秒換 60 次

In [ ]:
ticks = np.arange(12)
shown = np.floor(ticks * 90 / 60).astype(int)  # 每次換圖時，拿最新畫好的那一張
steps = np.diff(shown)                          # 相鄰兩次換圖，遊戲畫面前進了幾張
print("螢幕拿到第幾張：", shown.tolist())
print("每次前進幾張：", steps.tolist())

## 5. 不等換圖就把新畫面塞進去：撕裂

In [ ]:
W = 900
canvas = np.full((360, W, 3), 0.12)
x_old, x_new = 200, 200 + speed // 90           # 下一張畫面裡，貓咪往右移了 20 像素
tear = 150                                      # 螢幕由上往下更新到這一行時，新畫面到了
canvas[:tear, x_old:x_old + 360] = cat[:tear]
canvas[tear:, x_new:x_new + 360] = cat[tear:]
plt.figure(figsize=(10, 4)); plt.imshow(canvas); plt.axis("off"); plt.show()

---
覺得有趣的話，記得到 YouTube 訂閱 **像素腦**！有問題歡迎在影片留言 🐱